# 04 (PH) - Data preparation: aggregated tables for the station-level forecast

**Who worked on this notebook**

| Who | What |
| --- | --- |
| Peter Henskens | Decided the target (trips per station per day), the products, that they live as Postgres tables written from this notebook, and every threshold in part 0; created the schema and the write role; reviews every statement before it runs |
| Claude Code (AI assistant) | Drafts the SQL, the helpers and the checks; found the obstacles of the appendix by testing on 6 October 2026. Logged in `genai-log.md` |

**Goal.** Data preparation here means: from the view `citibike.trips` (Parquet through pg_duckdb) and the tables
`citibike.calendar_day` and `citibike.weather_daily`, build **aggregated tables** at the level the model needs, in the
schema `citibike_prep`. No features, no lags, no targets: those are modelling work for 05 onwards. Calendar and weather
already are tables and are joined there.

**Products**

| Table | Grain | What it is for |
| --- | --- | --- |
| `departures_raw`, `arrivals_raw` | raw station id x day | the aggregation of the 212 M trips, done once; everything below derives from these two without touching the view again |
| `station_labels_raw` | (role, raw id, name) | the station labels with lifetime and location, input of the station key |
| `station_id_map` | raw station id | which **station key** a raw id belongs to, and whether it is a New York station, another region or a system id |
| `station_dim` | station key | one row per physical station: ids, names, location, lifetime, trips, hotspot |
| `station_period` | station key x active period | when a station was in use (gaps longer than 60 days are "closed") |
| `station_day` | station key x active day | **the model's grain**: departures, arrivals, member and e-bike counts; 0 on a quiet day, no row where the station was not in use |
| `city_day` | day | city totals per day, dense (the two days without trips are rows with 0), the reference for the baseline of 03 §6.2 |

**How it is built**
1. Setup, thresholds, helpers.
2. Tables.
3. Raw aggregates from the view, through DuckDB: labels, departures, arrivals (per year), city totals.
4. The station key, from `station_labels_raw`: normalise, classify, link consecutive ids, hotspots.
5. `station_period` and `station_day`, in SQL on the server from the raw tables and the map.
6. Checks: every count reconciles with the view.
Appendix: what went wrong on 6 October and how each was solved.

**Status: written for review, nothing below has been executed yet** except the write-path test of August 2026 (appendix).

## 0. Setup, thresholds and helpers

Same connection as 02 and 03 (`citibike/connection.py`, login from the git-ignored `.env`); `sql()` is for reads.
Writing uses two helpers, both shaped by what the tests of 6 October showed (appendix):

- **`run()`** for DDL and `INSERT ... SELECT` between Postgres tables: one statement per transaction, opened
  **read-write** (the logins have `default_transaction_read_only = on`) and executed as the **write role**
  (`SET LOCAL ROLE`), so every table is owned by `citibike_prep_writer` and the default privileges give the teammates
  `SELECT`.
- **`load()`** for anything that reads the view: pg_duckdb 0.3 refuses `INSERT ... SELECT` when DuckDB executes the
  read, so the aggregation runs as SQL in DuckDB into a DataFrame (as the login, a member of `citibike_reader`), and
  `copy_into()` writes it with `COPY ... FROM STDIN` as the write role. `delete_where` removes the same period first,
  so every cell is repeatable. Measured on August 2026: 70.836 rows, query 3,8 s, copy 2,3 s.

Dates go into view queries as **literals**: that is how every view query in 02 and 03 ran; bound parameters on the
view are untested with pg_duckdb.

The constants are the **decisions** of this notebook, in one place so they are easy to challenge:

| Constant | Value | Why |
| --- | --- | --- |
| `LINK_MAX_M` | 100 m | Consecutive ids this close are one station. 02 §5.3: a renumbered station can shift ~80 m (`6789.04` -> `6789.20`); ids in use at the same time are never linked whatever the distance |
| `LINK_SAME_NAME_MAX_M` | 300 m | Consecutive ids with the **same normalised name** may be further apart (moved around the corner) |
| `OVERLAP_TOLERANCE_DAYS` | 7 | Two ids may overlap this long around a renumbering (switch-over days) and still count as consecutive |
| `HOTSPOT_M` | 50 m | Complete-linkage threshold of 02 §5.3 for the optional hotspot level (the map), not used by the model |
| `CLOSED_GAP_DAYS` | 60 | A station without any trip for longer than this was closed: those days get **no row**, not zeros (`6432.09` was gone for three years) |
| `FIRST_DAY` | 2021-01-01 | 185 trips start on 31 December 2020 (02 §3.1); left out everywhere |

In [ ]:
import io
import sys
import time

import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial import cKDTree
from sqlalchemy import text

sys.path.insert(0, "..")   # citibike/ -> connection.py (pg8000 driver, credentials from .env)
from connection import engine, sql, fmt, check_connection, notebook_style

check_connection()
notebook_style()

SCHEMA = "citibike_prep"
WRITER_ROLE = "citibike_prep_writer"

LINK_MAX_M = 100
LINK_SAME_NAME_MAX_M = 300
OVERLAP_TOLERANCE_DAYS = 7
HOTSPOT_M = 50
CLOSED_GAP_DAYS = 60
FIRST_DAY = "2021-01-01"
YEARS = range(2021, 2027)

# Degrees to metres on a flat projection around 40.73 N (accurate to well under 1 % over New York), as in 02 §5.3.
M_PER_DEG_LAT = 110_540
M_PER_DEG_LNG = 111_320 * np.cos(np.radians(40.73))


def run(statement: str) -> int:
    """Run one DDL or DML statement in its own read-write transaction, as the write role. Returns the row count."""
    t0 = time.time()
    with engine.begin() as conn:
        conn.execute(text("SET TRANSACTION READ WRITE"))        # the logins start read-only; this one writes
        conn.execute(text(f"SET LOCAL ROLE {WRITER_ROLE}"))     # objects end up owned by the role
        rows = conn.execute(text(statement)).rowcount
    print(f"{fmt(rows) if rows is not None and rows >= 0 else '-'} rows in {fmt(time.time() - t0, '.1f')} s")
    return rows


def copy_into(table: str, df: pd.DataFrame, columns: list[str], delete_where: str | None = None) -> None:
    """COPY a DataFrame into citibike_prep.<table> as the write role; delete_where first removes the same rows."""
    t0 = time.time()
    buf = io.BytesIO(df[columns].to_csv(index=False, header=False).encode())
    raw = engine.raw_connection()
    try:
        cur = raw.cursor()
        cur.execute("SET TRANSACTION READ WRITE")
        cur.execute(f"SET LOCAL ROLE {WRITER_ROLE}")
        if delete_where:
            cur.execute(f"DELETE FROM {SCHEMA}.{table} WHERE {delete_where}")
        cur.execute(f"COPY {SCHEMA}.{table} ({', '.join(columns)}) FROM STDIN WITH (FORMAT csv)", stream=buf)
        raw.commit()
    except Exception:
        raw.rollback()
        raise
    finally:
        raw.close()
    print(f"{fmt(len(df))} rows copied into {SCHEMA}.{table} ({fmt(len(buf.getvalue()) / 1e6, '.1f')} MB CSV) "
          f"in {fmt(time.time() - t0, '.1f')} s")


def load(table: str, columns: list[str], query: str, delete_where: str | None = None) -> pd.DataFrame:
    """SELECT through DuckDB (the view) into a DataFrame, then copy_into(). Returns the DataFrame."""
    df = sql(query)
    copy_into(table, df, columns, delete_where)
    return df

### 0.1 Who am I, and what may I do?
Expected: member of `citibike_reader` (DuckDB execution) and `citibike_prep_writer`; the schema owned by the write
role; `default_transaction_read_only = on` (that is why the helpers open their transactions read-write).

In [ ]:
sql(f"""
    SELECT current_user,
           (SELECT string_agg(r.rolname, ', ' ORDER BY r.rolname)
              FROM pg_auth_members m JOIN pg_roles r ON r.oid = m.roleid
             WHERE m.member = (SELECT oid FROM pg_roles WHERE rolname = current_user))      AS member_of,
           (SELECT pg_get_userbyid(nspowner) FROM pg_namespace WHERE nspname = '{SCHEMA}')  AS schema_owner,
           has_schema_privilege(current_user, '{SCHEMA}', 'CREATE')                         AS login_create,
           current_setting('default_transaction_read_only')                                 AS default_read_only
""").T

## 1. Tables

All `CREATE TABLE IF NOT EXISTS`, so the cell is repeatable. Explicit types; primary keys where the grain allows one
(`station_labels_raw` has nullable names and gets none; its loads delete per role instead). `station_day_raw` from the
write-path test of 6 October is dropped: it held departures only, under a name that promised both.

In [ ]:
run(f"DROP TABLE IF EXISTS {SCHEMA}.station_day_raw")

for statement in [
    f"""CREATE TABLE IF NOT EXISTS {SCHEMA}.departures_raw (
        day date NOT NULL, station_id text NOT NULL,
        departures integer NOT NULL, departures_member integer NOT NULL, departures_ebike integer NOT NULL,
        PRIMARY KEY (day, station_id))""",
    f"""CREATE TABLE IF NOT EXISTS {SCHEMA}.arrivals_raw (
        day date NOT NULL, station_id text NOT NULL,
        arrivals integer NOT NULL, arrivals_member integer NOT NULL, arrivals_ebike integer NOT NULL,
        PRIMARY KEY (day, station_id))""",
    f"""CREATE TABLE IF NOT EXISTS {SCHEMA}.station_labels_raw (
        role text NOT NULL, station_id text NOT NULL, station_name text,
        first_seen timestamp NOT NULL, last_seen timestamp NOT NULL,
        trips bigint NOT NULL, classic_trips bigint NOT NULL, located_trips bigint NOT NULL,
        lat_classic double precision, lng_classic double precision, lat_all double precision, lng_all double precision)""",
    f"""CREATE TABLE IF NOT EXISTS {SCHEMA}.station_id_map (
        station_id text PRIMARY KEY, station_id_norm text NOT NULL, station_key text NOT NULL,
        kind text NOT NULL CHECK (kind IN ('nyc', 'other region', 'system')))""",
    f"""CREATE TABLE IF NOT EXISTS {SCHEMA}.station_dim (
        station_key text PRIMARY KEY, current_id text NOT NULL, station_ids text[] NOT NULL,
        name text, names text, lat double precision, lng double precision,
        first_seen timestamp NOT NULL, last_seen timestamp NOT NULL,
        trips_as_start bigint NOT NULL, trips_as_end bigint NOT NULL, hotspot_id integer)""",
    f"""CREATE TABLE IF NOT EXISTS {SCHEMA}.station_period (
        station_key text NOT NULL, period_no integer NOT NULL, first_day date NOT NULL, last_day date NOT NULL,
        PRIMARY KEY (station_key, period_no))""",
    f"""CREATE TABLE IF NOT EXISTS {SCHEMA}.station_day (
        station_key text NOT NULL, day date NOT NULL, period_no integer NOT NULL,
        departures integer NOT NULL, departures_member integer NOT NULL, departures_ebike integer NOT NULL,
        arrivals integer NOT NULL, arrivals_member integer NOT NULL, arrivals_ebike integer NOT NULL,
        PRIMARY KEY (station_key, day))""",
    f"""CREATE TABLE IF NOT EXISTS {SCHEMA}.city_day (
        day date PRIMARY KEY, trips integer NOT NULL, trips_member integer NOT NULL, trips_ebike integer NOT NULL,
        trips_without_start_station integer NOT NULL, trips_without_end_station integer NOT NULL,
        no_trips boolean NOT NULL)""",
]:
    run(statement)

## 2. Raw aggregates from the view

The only part that reads the 212 M trips. Four aggregates, each through `load()`.

### 2.1 Station labels, as start and as end station
Per role and (raw id, name): first and last trip, trips, and the location. For a classic bike the coordinates are
those of the dock (02 §5.3: spread 0 m for 90 % of the keys), for an e-bike the GPS position of the bike, so the
location is the classic-bike average, with the all-trips average as fallback. Coordinates outside the New York box
(for example 0, 0) are ignored. Null names are kept as their own label; the canonical name is chosen in part 3 from the
non-null ones. The **end** side is new compared with 02: `station_day` counts arrivals too, and some ids only occur as
an end station. Two loads (02 §5.3 took about two minutes for the start side alone).

In [ ]:
def labels_query(role: str) -> str:
    id_, name = f"{role}_station_id", f"{role}_station_name"
    lat, lng = f"{role}_lat", f"{role}_lng"
    when = "started_at" if role == "start" else "ended_at"
    in_nyc = f"{lat} BETWEEN 40 AND 41.5 AND {lng} BETWEEN -75 AND -73"
    classic = f"rideable_type = 'classic_bike' AND {in_nyc}"
    return f"""
        SELECT '{role}'                                   AS role,
               {id_}                                      AS station_id,
               {name}                                     AS station_name,
               min({when})                                AS first_seen,
               max({when})                                AS last_seen,
               count(*)                                   AS trips,
               count(*) FILTER (WHERE {classic})          AS classic_trips,
               count(*) FILTER (WHERE {in_nyc})           AS located_trips,
               avg({lat}) FILTER (WHERE {classic})        AS lat_classic,
               avg({lng}) FILTER (WHERE {classic})        AS lng_classic,
               avg({lat}) FILTER (WHERE {in_nyc})         AS lat_all,
               avg({lng}) FILTER (WHERE {in_nyc})         AS lng_all
        FROM citibike.trips
        WHERE started_at >= '{FIRST_DAY}' AND {id_} IS NOT NULL
        GROUP BY 1, 2, 3
    """

LABEL_COLUMNS = ["role", "station_id", "station_name", "first_seen", "last_seen", "trips", "classic_trips", "located_trips",
                 "lat_classic", "lng_classic", "lat_all", "lng_all"]
for role in ["start", "end"]:
    load("station_labels_raw", LABEL_COLUMNS, labels_query(role), delete_where=f"role = '{role}'")

### 2.2 Departures per raw start id and day, per year
Grouped by the **raw** id; the normalisation and the mapping to the station key happen on this table (part 3 and 5),
never on the trips. One load per year keeps every query well inside the 10-minute statement timeout and makes the
progress visible. Rows without a start station (0,04 % of the trips, 02 §2.2) have no station and are not here; the
city totals of 2.4 count them.

In [ ]:
def demand_query(role: str, year: int) -> str:
    id_ = f"{role}_station_id"
    when = "started_at" if role == "start" else "ended_at"
    what = "departures" if role == "start" else "arrivals"
    return f"""
        SELECT {when}::date                                               AS day,
               {id_}                                                      AS station_id,
               count(*)                                                   AS {what},
               count(*) FILTER (WHERE member_casual = 'member')           AS {what}_member,
               count(*) FILTER (WHERE rideable_type = 'electric_bike')    AS {what}_ebike
        FROM citibike.trips
        WHERE started_at >= '{year}-01-01' AND started_at < '{year + 1}-01-01' AND {id_} IS NOT NULL
        GROUP BY 1, 2
    """

DEP_COLUMNS = ["day", "station_id", "departures", "departures_member", "departures_ebike"]
for year in YEARS:
    print(f"departures {year}: ", end="")
    load("departures_raw", DEP_COLUMNS, demand_query("start", year),
         delete_where=f"day >= '{year}-01-01' AND day < '{year + 1}-01-01'")

### 2.3 Arrivals per raw end id and day, per year
The day of an arrival is the day the trip **ended** (a trip from 23:50 to 00:10 arrives the next day). The year filter
stays on `started_at`, so every monthly file is read once, but then a year's query also returns arrivals on 1 January
of the **next** year (trips that started on 31 December). Two years' queries therefore share rows for 1 January, and
the six results are summed per (day, station) and copied in one go after a truncate; a rerun of this cell is a full
rerun. Trips without an end station (0,24 %: e-bikes parked outside a dock, 02 §2.2) have no arrival row.

In [ ]:
ARR_COLUMNS = ["day", "station_id", "arrivals", "arrivals_member", "arrivals_ebike"]
run(f"TRUNCATE {SCHEMA}.arrivals_raw")
arrivals_parts = []
for year in YEARS:
    print(f"arrivals {year}: ", end="")
    arrivals_parts.append(sql(demand_query("end", year)))
# Two years' queries can both contain 1 January rows of the same (day, station): sum them before the copy.
arrivals = (pd.concat(arrivals_parts, ignore_index=True)
            .groupby(["day", "station_id"], as_index=False)[["arrivals", "arrivals_member", "arrivals_ebike"]].sum())
copy_into("arrivals_raw", arrivals, ARR_COLUMNS)
del arrivals_parts

### 2.4 City totals per day
Trips per day for the whole city, including the trips without a station, by user type and bike type, plus how many
trips had no start or no end station (the reconciliation in part 6 uses them). Then the days without any trip
(02 §3.4: 2 February 2021 and 23 February 2026, both in a snowstorm) are added as rows with 0 and `no_trips = true`,
so the table is dense from `FIRST_DAY` to the last day with data.

In [ ]:
CITY_COLUMNS = ["day", "trips", "trips_member", "trips_ebike", "trips_without_start_station", "trips_without_end_station", "no_trips"]
city = load("city_day", CITY_COLUMNS, f"""
    SELECT started_at::date                                              AS day,
           count(*)                                                      AS trips,
           count(*) FILTER (WHERE member_casual = 'member')              AS trips_member,
           count(*) FILTER (WHERE rideable_type = 'electric_bike')       AS trips_ebike,
           count(*) FILTER (WHERE start_station_id IS NULL)              AS trips_without_start_station,
           count(*) FILTER (WHERE end_station_id IS NULL)                AS trips_without_end_station,
           false                                                         AS no_trips
    FROM citibike.trips
    WHERE started_at >= '{FIRST_DAY}'
    GROUP BY 1
""", delete_where="true")

filled = run(f"""
    INSERT INTO {SCHEMA}.city_day (day, trips, trips_member, trips_ebike, trips_without_start_station, trips_without_end_station, no_trips)
    SELECT d::date, 0, 0, 0, 0, 0, true
    FROM generate_series(DATE '{FIRST_DAY}', (SELECT max(day) FROM {SCHEMA}.city_day), interval '1 day') AS d
    WHERE NOT EXISTS (SELECT 1 FROM {SCHEMA}.city_day c WHERE c.day = d::date)
""")
print(f"days without any trip added: {filled} (02 §3.4 found 2)")

## 3. The station key

### 3.1 Why
02 §5.3 showed that neither `station_id` nor `station_name` identifies a physical station over time: 2.921 (id, name)
labels for 2.794 ids and 2.746 names, with four kinds of noise: **the same id in two formats** (`7386.10` and `7386.1`,
a trailing zero lost somewhere), **renumbering** (50 names moved to a new id on the same spot, `11 St & 43 Ave`:
`6438.04` -> `6438.05` in January 2025), **renaming** (98 ids, `Pershing Square North` -> `Park Ave & E 42 St`) and
**non-stations** (`Lab - NYC`, `SYS032`, 540 trips with Jersey City / Hoboken ids). Grouping by the raw id would make
a renumbered station *die* and a *new* one appear with no history, exactly where the model needs its lags.

**The rule** (02 §5.3): a *station* is a normalised id, with consecutive ids on the same spot linked into one; ids in
use **at the same time** are different stations however close they are (`Park Ave & E 41 St` and `E 42 St`, 37 m).
A group of stations within 50 m is a *hotspot*, an optional level for the map.

This is the one part that runs in pandas: the linking is a union-find over a few thousand rows, readable here and
unreadable as a recursive CTE. Input is `station_labels_raw` (2.1), output goes into `station_id_map` and `station_dim`.

### 3.2 Normalise and classify
Citi Bike ids are `NNNN.NN`. Two things happened to them in the files, both found on the loaded labels (6 October).
1. **A lost trailing zero**: one decimal means `.N0` (`7386.1` is `7386.10`; a float cannot lose a leading zero, so
   `.05` always survives). 119 raw ids have that form, nine of them without any two-decimal twin (`6131.1`, 250.404
   trips), and it occurs over the whole period, not only 2023 - 2025 as 02 §5.3 assumed.
2. **A suffix**: `5308.04_`, `5348.06_old`, `6912.01_Pillar` are the same station as `5308.04`, `5348.06`, `6912.01`: an
   extra or temporary dock, 0 to 33 m from the base id and in use in parallel with it (`_old` is the old dock during a
   move, 291 m). Nineteen such ids, about 73.000 trips, that a strict "numeric only" rule would have thrown away as
   system ids.

Rules: strip the suffix, then append the zero. The check after both: the forms of one normalised id must be on the
same spot. Pairs more than 25 m apart are listed for review (a station that moved: `5235.10`, 59 m, same name); a pair
more than 500 m apart would mean two different stations and stops the notebook.

Then every normalised id is **New York** (`NNNN.NN`), **other region** (`JC...`, `HB...`) or **system** (`SYS0xx`,
`Lab - NYC`, `Shop Morgan`, demo and test ids). The last two are kept in the map with their kind, so the counts of
part 5 add up, and left out of `station_dim` and `station_day`.

In [15]:
labels = sql(f"SELECT * FROM {SCHEMA}.station_labels_raw")
labels["first_seen"] = pd.to_datetime(labels.first_seen)
labels["last_seen"] = pd.to_datetime(labels.last_seen)

PATTERNS = {"two decimals (NNNN.NN)": r"\d+\.\d\d", "one decimal (NNNN.N)": r"\d+\.\d",
            "numeric with suffix (NNNN.NN_...)": r"\d+\.\d+_.*", "other region (JC..., HB...)": r"(JC|HB).*"}

def normalise_id(ids: pd.Series) -> pd.Series:
    s = ids.str.strip()
    s = s.str.replace(r"^(\d+\.\d+)_.*$", r"\1", regex=True)     # 5308.04_ and 6912.01_Pillar -> 5308.04, 6912.01
    return s.mask(s.str.fullmatch(r"\d+\.\d"), s + "0")           # 7386.1 -> 7386.10

def classify(norm_ids: pd.Series) -> pd.Series:
    return pd.Series(np.select([norm_ids.str.fullmatch(r"\d+\.\d\d"), norm_ids.str.fullmatch(r"(JC|HB).*")],
                               ["nyc", "other region"], "system"), index=norm_ids.index)

def metres(lat1, lng1, lat2, lng2):
    return np.hypot((lat1 - lat2) * M_PER_DEG_LAT, (lng1 - lng2) * M_PER_DEG_LNG)

labels["id_pattern"] = pd.Series(np.select([labels.station_id.str.strip().str.fullmatch(p) for p in PATTERNS.values()],
                                           list(PATTERNS), "system / text"), index=labels.index)
labels["station_id_norm"] = normalise_id(labels.station_id)
labels["kind"] = classify(labels.station_id_norm)
print("Id patterns before normalisation (both roles):")
display(labels.groupby("id_pattern").agg(labels=("station_id", "size"), raw_ids=("station_id", "nunique"), trips=("trips", "sum")))

# Check: the forms of one normalised id are the same spot. Location per raw form from the classic-bike trips,
# falling back to all located trips for forms without any (e-bike-only docks such as the Pillar pilots).
has_classic = labels.classic_trips > 0
w = labels.classic_trips.where(has_classic, labels.located_trips)
lat = labels.lat_classic.where(has_classic, labels.lat_all)
lng = labels.lng_classic.where(has_classic, labels.lng_all)
forms = pd.DataFrame({"w": w, "lat_w": lat * w, "lng_w": lng * w,
                      "norm": labels.station_id_norm, "raw": labels.station_id}).groupby(["norm", "raw"]).sum()
forms["lat"], forms["lng"] = forms.lat_w / forms.w, forms.lng_w / forms.w
multi = forms[forms.groupby(level=0).w.transform("size") > 1].reset_index()
check = multi.groupby("norm").agg(forms=("raw", list), lat_min=("lat", "min"), lat_max=("lat", "max"),
                                  lng_min=("lng", "min"), lng_max=("lng", "max"))
check["distance_m"] = metres(check.lat_min, check.lng_min, check.lat_max, check.lng_max).round(1)
check["names"] = labels.groupby("station_id_norm").station_name.apply(lambda s: " | ".join(dict.fromkeys(s.dropna()))).reindex(check.index)
far = check[check.distance_m > 25].sort_values("distance_m", ascending=False)
print(f"{fmt(len(check))} normalised ids have more than one raw form; {fmt(len(far))} of them lie more than 25 m apart "
      f"(a station that moved or a second dock):")
display(far[["forms", "names", "distance_m"]])
assert (check.distance_m.fillna(0) < 500).all(), "a merged pair is further than 500 m apart: two stations, inspect before continuing"
labels.groupby("kind").agg(labels=("station_id", "size"), ids=("station_id_norm", "nunique"), trips=("trips", "sum"))

5.884 rows in 1,0 s
Id patterns before normalisation (both roles):


,labels,raw_ids,trips
id_pattern,,,
numeric with suffix (NNNN.NN_...),38,19,73.289
one decimal (NNNN.N),245,119,3.654.797
"other region (JC..., HB...)",188,104,13.353
system / text,47,22,27.977
two decimals (NNNN.NN),5.366,2.566,420.021.294


128 normalised ids have more than one raw form; 4 of them lie more than 25 m apart (a station that moved or a second dock):


,forms,names,distance_m
norm,,,
5348.06,"['5348.06', '5348.06_old']",Kent Ave & N 3 St,"290,7"
5235.10,"['5235.1', '5235.10']",Driggs Ave & S 2 St,"58,8"
6569.09,"['6569.09', '6569.09_']",W 35 St & 9 Ave,"32,9"
5820.08,"['5820.08', '5820.08_Pillar']",E 13 St & 2 Ave | Pillar E 13 St & 2 Ave,"29,7"


,labels,ids,trips
kind,,,
nyc,5.649,2.575,423.749.380
other region,188,104,13.353
system,47,22,27.977


### 3.3 One row per normalised id
Names are labels: the name with the most trips (both roles) becomes the canonical name, every spelling is kept. The
location is the classic-bike average over both roles (fallback: all located trips), the lifetime the first and last
trip in either role. The same id with a long gap (`6432.09`: 2021, then again from December 2024) stays one row; the
gap is handled by `station_period` in part 5. The name normalisation (lower case, `&` -> `and`, punctuation out,
`Avenue` -> `Ave`, `60th` -> `60`) is the one from 02 §5.3 and only serves the same-name rule in 3.4.

In [16]:
ABBREVIATIONS = {r"\bavenue\b": "ave", r"\bstreet\b": "st", r"\bplace\b": "pl", r"\bboulevard\b": "blvd",
                 r"\bparkway\b": "pkwy", r"\b(\d+)(st|nd|rd|th)\b": r"\1"}

def normalise_name(names: pd.Series) -> pd.Series:
    out = (names.fillna("").str.lower().str.replace("&", " and ", regex=False)
           .str.replace(r"[^a-z0-9 ]", " ", regex=True))
    for pattern, short in ABBREVIATIONS.items():
        out = out.str.replace(pattern, short, regex=True)
    return out.str.split().str.join(" ")

def weighted_coords(df: pd.DataFrame, lat: str, lng: str, weight: str) -> pd.DataFrame:
    w = df[weight]
    g = pd.DataFrame({"w": w, "lat_w": df[lat] * w, "lng_w": df[lng] * w, "key": df.station_id_norm}).groupby("key").sum()
    return pd.DataFrame({"lat": g.lat_w / g.w, "lng": g.lng_w / g.w})   # NaN where the weight is 0

by_id = labels.groupby("station_id_norm")
stations = pd.DataFrame({
    "first_seen": by_id.first_seen.min(), "last_seen": by_id.last_seen.max(), "kind": by_id.kind.first(),
    "trips_as_start": labels[labels.role == "start"].groupby("station_id_norm").trips.sum(),
    "trips_as_end": labels[labels.role == "end"].groupby("station_id_norm").trips.sum(),
    "raw_ids": by_id.station_id.apply(lambda s: sorted(set(s))),
})
stations.index.name = "station_id_norm"
stations[["trips_as_start", "trips_as_end"]] = stations[["trips_as_start", "trips_as_end"]].fillna(0).astype("int64")
classic = weighted_coords(labels, "lat_classic", "lng_classic", "classic_trips").reindex(stations.index)
located = weighted_coords(labels, "lat_all", "lng_all", "located_trips").reindex(stations.index)
stations["lat"], stations["lng"] = classic.lat.fillna(located.lat), classic.lng.fillna(located.lng)
stations["coords_from"] = np.where(classic.lat.notna(), "classic bikes", np.where(located.lat.notna(), "all trips", "none"))

by_name = (labels.dropna(subset=["station_name"]).groupby(["station_id_norm", "station_name"]).trips.sum()
           .reset_index().sort_values(["station_id_norm", "trips"], ascending=[True, False]))
stations["name"] = by_name.drop_duplicates("station_id_norm").set_index("station_id_norm").station_name
stations["names"] = by_name.groupby("station_id_norm").station_name.apply(" | ".join)
stations["name_norm"] = normalise_name(stations.name)
stations["x"], stations["y"] = stations.lng * M_PER_DEG_LNG, stations.lat * M_PER_DEG_LAT

print(f"{fmt(len(stations))} normalised ids")
stations.groupby(["kind", "coords_from"]).agg(ids=("name", "size"), trips_as_start=("trips_as_start", "sum"))

2.701 normalised ids


ids  trips_as_start
kind         coords_from                        
nyc          classic bikes  2575       212092181
other region all trips        14              11
             classic bikes    90             529
system       all trips         3               3
             classic bikes    15           10746
             none              4              41

### 3.4 Link consecutive ids into one station
Candidate pairs: New York ids within `LINK_SAME_NAME_MAX_M` of each other (k-d tree). A pair is linked when the ids
are **consecutive in time** (one's last trip at most `OVERLAP_TOLERANCE_DAYS` after the other's first trip) **and**
closer than `LINK_MAX_M`, or closer than `LINK_SAME_NAME_MAX_M` with the same normalised name. Ids that overlap longer
are never linked: in use at the same time means two stations. Chains (A -> B -> C) through union-find.

The **station key** is the **earliest** id of the chain, as text: `6438.04` stays the key after `6438.05` joined. A
monthly rebuild appends new ids at the end and never changes the first one, so the key is stable without a lookup
table of its own; a sequence number could shift, the latest id changes at every renumbering.

The sensitivity table gives the number of stations at 50, 100 and 150 m, as 02 §5.3 did for the places. The spot
checks are the cases 02 documented, and fail loudly if the rule does not reproduce them.

In [17]:
def link_stations(st: pd.DataFrame, max_m: float, same_name_max_m: float, tolerance_days: int) -> pd.Series:
    """Map every normalised id to its station key (the earliest id of its chain). Non-NYC ids map to themselves."""
    nyc = st[(st.kind == "nyc") & st.lat.notna()]
    ids = nyc.index.to_numpy()
    xy = nyc[["x", "y"]].to_numpy()
    pairs = cKDTree(xy).query_pairs(r=max(max_m, same_name_max_m), output_type="ndarray")
    a, b = ids[pairs[:, 0]], ids[pairs[:, 1]]
    d = np.hypot(xy[pairs[:, 0], 0] - xy[pairs[:, 1], 0], xy[pairs[:, 0], 1] - xy[pairs[:, 1], 1])
    tol = pd.Timedelta(days=tolerance_days)
    first, last = nyc.first_seen, nyc.last_seen
    consecutive = ((last.loc[a].to_numpy() <= first.loc[b].to_numpy() + tol)
                   | (last.loc[b].to_numpy() <= first.loc[a].to_numpy() + tol))
    same_name = nyc.name_norm.loc[a].to_numpy() == nyc.name_norm.loc[b].to_numpy()
    link = consecutive & ((d <= max_m) | (same_name & (d <= same_name_max_m)))

    parent = {i: i for i in st.index}
    def root(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    for i, j in zip(a[link], b[link]):
        ri, rj = root(i), root(j)
        if ri != rj:
            parent[rj] = ri
    group = pd.Series({i: root(i) for i in st.index}, name="group")
    earliest = (st.assign(group=group).reset_index().sort_values(["first_seen", "station_id_norm"])
                .drop_duplicates("group").set_index("group").station_id_norm)
    return group.map(earliest).rename("station_key")

sensitivity = pd.DataFrame({"link_max_m": [50, 100, 150]})
sensitivity["stations"] = [link_stations(stations, m, LINK_SAME_NAME_MAX_M, OVERLAP_TOLERANCE_DAYS)[stations.kind == "nyc"].nunique()
                           for m in sensitivity.link_max_m]
display(sensitivity)

stations["station_key"] = link_stations(stations, LINK_MAX_M, LINK_SAME_NAME_MAX_M, OVERLAP_TOLERANCE_DAYS)
chain = stations[stations.kind == "nyc"].groupby("station_key").size()
print(f"{fmt((stations.kind == 'nyc').sum())} New York ids -> {fmt(len(chain))} stations at {LINK_MAX_M} m; "
      f"{fmt((chain > 1).sum())} stations have more than one id (longest chain: {chain.max()} ids)")

key = stations.station_key
for a, b in [("7386.10", "7386.1"), ("6438.04", "6438.05"), ("6789.04", "6789.20"), ("6789.20", "6789.2"), ("6278.04", "6319.03")]:
    a, b = normalise_id(pd.Series([a, b]))
    assert key.get(a) == key.get(b), f"{a} and {b} should be one station"
for a, b in [("6432.08", "6432.09"), ("6233.04", "6306.07")]:
    assert key.get(a) != key.get(b), f"{a} and {b} should stay two stations"
print("spot checks of 02 §5.3 passed")
(stations[stations.kind == "nyc"].reset_index().groupby("station_key")
 .agg(ids=("station_id_norm", list), names=("name", lambda s: " | ".join(dict.fromkeys(s.dropna()))),
      first_seen=("first_seen", "min"), last_seen=("last_seen", "max"))
 .assign(n_ids=lambda d: d.ids.str.len()).sort_values("n_ids", ascending=False).head(8))

,link_max_m,stations
0,50,2.502
1,100,2.455
2,150,2.418


2.575 New York ids -> 2.455 stations at 100 m; 112 stations have more than one id (longest chain: 4 ids)
spot checks of 02 §5.3 passed


,ids,names,first_seen,last_seen,n_ids
station_key,,,,,
7596.10,"['7563.07', '7579.11', '7596.10', '7596.11']",E 116 St & 1 Ave | E 117 St & 1 Ave | E 118 St & 1 Ave | 1 Ave & E 118 St,2021-01-01 00:13:33.636000,2026-08-31 23:56:10.492000,4
3460.04,"['3460.04', '3460.05', '3460.06']",2 Av & 37 St | 2 Ave & 39 St | 2 Ave & 37 St,2021-02-21 15:08:00.909000,2025-08-20 21:02:08.136000,3
6076.01,"['6076.01', '6076.02', '6151.05']",48 Ave & 30 Pl | 30 St & 48 Ave | 48 Ave & 31 St,2021-01-01 07:00:54.728000,2026-08-31 21:13:28.769000,3
5329.06,"['5329.06', '5329.08', '5329.13']",Murray St & West St | Murray St & West St (NE Corner),2021-01-01 10:26:49.974000,2026-08-31 23:47:28.226000,3
4962.04,"['4962.02', '4962.04', '4962.09']",Whitehall St & Bridge St | Water - Whitehall Plaza | Pearl St & Whitehall St,2021-01-01 00:00:35.365000,2026-08-31 23:47:04.021000,3
6441.05,"['6441.05', '6441.06', '6560.15']",Broadway & W 38 St | W 37 St & Broadway,2021-01-01 00:07:15,2026-08-31 23:29:29.444000,3
6789.04,"['6789.04', '6789.08', '6789.20']",31 St & Broadway | Broadway & 31 St,2021-01-01 00:18:55.581000,2026-08-31 23:55:04.747000,3
5073.07,"['5073.07', '5073.08']",Greenwich St & Rector St,2022-01-27 11:57:52.769000,2026-08-31 23:42:16.190000,2


### 3.5 Hotspots, and the two tables
The hotspot is the 50 m complete-linkage clustering of 02 §5.3 on one point per station (every pair inside a hotspot
closer than 50 m, so a row of neighbours cannot chain into one). `station_dim` holds the New York stations only;
`station_id_map` holds **every** raw id (both formats) with its kind, so a trip maps with a plain join.

In [18]:
nyc = stations[stations.kind == "nyc"].reset_index()
flat = nyc.sort_values("last_seen").drop_duplicates("station_key", keep="last").set_index("station_key")
busiest = nyc.sort_values("trips_as_start").drop_duplicates("station_key", keep="last").set_index("station_key")
wt = nyc.trips_as_start.clip(lower=1)
dim = pd.DataFrame({
    "current_id": flat.station_id_norm,
    "station_ids": nyc.groupby("station_key").station_id_norm.apply(sorted),
    "name": busiest.name,
    "names": nyc.groupby("station_key").names.apply(lambda s: " | ".join(dict.fromkeys(n for x in s.dropna() for n in x.split(" | ")))),
    "lat": (nyc.lat * wt).groupby(nyc.station_key).sum() / wt.groupby(nyc.station_key).sum(),
    "lng": (nyc.lng * wt).groupby(nyc.station_key).sum() / wt.groupby(nyc.station_key).sum(),
    "first_seen": nyc.groupby("station_key").first_seen.min(), "last_seen": nyc.groupby("station_key").last_seen.max(),
    "trips_as_start": nyc.groupby("station_key").trips_as_start.sum(), "trips_as_end": nyc.groupby("station_key").trips_as_end.sum(),
})
dim.index.name = "station_key"
with_coords = dim.lat.notna()
tree = linkage(np.column_stack([dim.lng[with_coords] * M_PER_DEG_LNG, dim.lat[with_coords] * M_PER_DEG_LAT]), method="complete")
dim["hotspot_id"] = pd.Series(fcluster(tree, HOTSPOT_M, criterion="distance"), index=dim.index[with_coords]).reindex(dim.index).astype("Int64")
print(f"station_dim: {fmt(len(dim))} stations, {fmt(dim.hotspot_id.nunique())} hotspots, "
      f"{fmt(int((~with_coords).sum()))} without coordinates")

id_map = (labels[["station_id", "station_id_norm"]].drop_duplicates("station_id")
          .assign(station_key=lambda d: d.station_id_norm.map(stations.station_key),
                  kind=lambda d: d.station_id_norm.map(stations.kind)))
assert id_map.station_key.notna().all() and id_map.kind.notna().all()

dim_rows = dim.reset_index()
dim_rows["station_ids"] = dim_rows.station_ids.apply(lambda ids: "{" + ",".join(ids) + "}")   # Postgres array literal
copy_into("station_id_map", id_map, ["station_id", "station_id_norm", "station_key", "kind"], delete_where="true")
copy_into("station_dim", dim_rows, ["station_key", "current_id", "station_ids", "name", "names", "lat", "lng",
                                    "first_seen", "last_seen", "trips_as_start", "trips_as_end", "hotspot_id"],
          delete_where="true")

station_dim: 2.455 stations, 2.434 hotspots, 0 without coordinates
2.830 rows copied into citibike_prep.station_id_map (0,1 MB CSV) in 0,5 s
2.455 rows copied into citibike_prep.station_dim (0,4 MB CSV) in 0,8 s


## 4. `station_period` and `station_day`, in SQL on the server

From here on no DuckDB: the sources are the raw tables and the map, so a plain `INSERT ... SELECT` works and the
data stays on the server.

### 4.1 Active periods
Per station: the days with at least one departure or arrival (both raw tables, mapped through `station_id_map`,
New York stations only), in order; a gap longer than `CLOSED_GAP_DAYS` starts a new period (gaps-and-islands with
`lag()` and a running sum). Inside a period every day gets a row in `station_day`, outside none. Decision: a gap of
more than 60 days is read as "closed", because a row of zeros there would teach the model a quiet that never existed.

In [19]:
run(f"TRUNCATE {SCHEMA}.station_period")
run(f"""
    INSERT INTO {SCHEMA}.station_period (station_key, period_no, first_day, last_day)
    SELECT station_key,
           row_number() OVER (PARTITION BY station_key ORDER BY min(day)) AS period_no,
           min(day) AS first_day, max(day) AS last_day
    FROM (
        SELECT station_key, day, sum(new_period) OVER (PARTITION BY station_key ORDER BY day) AS period_grp
        FROM (
            SELECT station_key, day,
                   CASE WHEN day - lag(day) OVER (PARTITION BY station_key ORDER BY day) > {CLOSED_GAP_DAYS} THEN 1 ELSE 0 END AS new_period
            FROM (
                SELECT DISTINCT m.station_key, r.day
                FROM (SELECT day, station_id FROM {SCHEMA}.departures_raw
                      UNION SELECT day, station_id FROM {SCHEMA}.arrivals_raw) r
                JOIN {SCHEMA}.station_id_map m ON m.station_id = r.station_id
                WHERE m.kind = 'nyc'
            ) active
        ) flagged
    ) grouped
    GROUP BY station_key, period_grp
""")
sql(f"""
    SELECT count(*) AS periods, count(DISTINCT station_key) AS stations,
           count(*) FILTER (WHERE period_no > 1) AS reopenings,
           sum(last_day - first_day + 1) AS active_station_days
    FROM {SCHEMA}.station_period
""")

- rows in 0,3 s
2.720 rows in 38,6 s
1 rows in 0,4 s


,periods,stations,reopenings,active_station_days
0,2.720,2.455,265,3.872.821


### 4.2 The dense station-day table, per year
For every active period, every day of the year inside it (`generate_series`), left-joined with the departures and
arrivals summed per station key and day (the two formats of one id, or a renumbered pair on its switch-over day,
become one row); `coalesce(..., 0)` makes a quiet day a 0. One `INSERT` per year keeps each statement small.

In [20]:
run(f"TRUNCATE {SCHEMA}.station_day")
for year in YEARS:
    first, last = f"DATE '{year}-01-01'", f"DATE '{year}-12-31'"
    print(f"station_day {year}: ", end="")
    run(f"""
        INSERT INTO {SCHEMA}.station_day (station_key, day, period_no, departures, departures_member, departures_ebike,
                                          arrivals, arrivals_member, arrivals_ebike)
        SELECT p.station_key, d.day::date, p.period_no,
               coalesce(dep.departures, 0), coalesce(dep.departures_member, 0), coalesce(dep.departures_ebike, 0),
               coalesce(arr.arrivals, 0), coalesce(arr.arrivals_member, 0), coalesce(arr.arrivals_ebike, 0)
        FROM {SCHEMA}.station_period p
        CROSS JOIN LATERAL generate_series(greatest(p.first_day, {first}), least(p.last_day, {last}), interval '1 day') AS d(day)
        LEFT JOIN (
            SELECT m.station_key, r.day, sum(r.departures) AS departures,
                   sum(r.departures_member) AS departures_member, sum(r.departures_ebike) AS departures_ebike
            FROM {SCHEMA}.departures_raw r JOIN {SCHEMA}.station_id_map m ON m.station_id = r.station_id
            WHERE m.kind = 'nyc' AND r.day BETWEEN {first} AND {last}
            GROUP BY 1, 2
        ) dep ON dep.station_key = p.station_key AND dep.day = d.day::date
        LEFT JOIN (
            SELECT m.station_key, r.day, sum(r.arrivals) AS arrivals,
                   sum(r.arrivals_member) AS arrivals_member, sum(r.arrivals_ebike) AS arrivals_ebike
            FROM {SCHEMA}.arrivals_raw r JOIN {SCHEMA}.station_id_map m ON m.station_id = r.station_id
            WHERE m.kind = 'nyc' AND r.day BETWEEN {first} AND {last}
            GROUP BY 1, 2
        ) arr ON arr.station_key = p.station_key AND arr.day = d.day::date
        WHERE p.last_day >= {first} AND p.first_day <= {last}
    """)

- rows in 0,3 s
station_day 2021: 513.920 rows in 9,2 s
station_day 2022: 572.627 rows in 10,7 s
station_day 2023: 683.874 rows in 14,2 s
station_day 2024: 776.861 rows in 17,7 s
station_day 2025: 782.694 rows in 17,4 s
station_day 2026: 542.845 rows in 12,7 s


## 5. Checks

Every number must reconcile with the view, otherwise a product silently lost or doubled trips:
- departures in `station_day` + departures of non-NYC ids = all departures in `departures_raw`, and `departures_raw`
  + trips without a start station (`city_day`) = all trips (`city_day`); the same for arrivals;
- `station_day` has exactly as many rows as the active periods promise, no row outside a period (by construction), and
  every New York station of `station_dim` appears;
- the distribution of departures per station-day and the share of quiet days, to see what the model will face.

In [21]:
totals = sql(f"""
    SELECT (SELECT sum(departures) FROM {SCHEMA}.station_day)                                                   AS sd_departures,
           (SELECT sum(arrivals) FROM {SCHEMA}.station_day)                                                     AS sd_arrivals,
           (SELECT sum(r.departures) FROM {SCHEMA}.departures_raw r JOIN {SCHEMA}.station_id_map m USING (station_id)
             WHERE m.kind <> 'nyc')                                                                            AS dep_not_nyc,
           (SELECT sum(r.arrivals) FROM {SCHEMA}.arrivals_raw r JOIN {SCHEMA}.station_id_map m USING (station_id)
             WHERE m.kind <> 'nyc')                                                                            AS arr_not_nyc,
           (SELECT sum(departures) FROM {SCHEMA}.departures_raw)                                                AS raw_departures,
           (SELECT sum(arrivals) FROM {SCHEMA}.arrivals_raw)                                                    AS raw_arrivals,
           (SELECT sum(trips) FROM {SCHEMA}.city_day)                                                           AS trips,
           (SELECT sum(trips_without_start_station) FROM {SCHEMA}.city_day)                                     AS no_start,
           (SELECT sum(trips_without_end_station) FROM {SCHEMA}.city_day)                                       AS no_end
""").iloc[0].fillna(0).astype("int64")
display(totals.to_frame("count"))
assert totals.sd_departures + totals.dep_not_nyc == totals.raw_departures, "departures: station_day + non-NYC != raw"
assert totals.sd_arrivals + totals.arr_not_nyc == totals.raw_arrivals, "arrivals: station_day + non-NYC != raw"
assert totals.raw_departures + totals.no_start == totals.trips, "departures_raw + trips without start station != all trips"
assert totals.raw_arrivals + totals.no_end == totals.trips, "arrivals_raw + trips without end station != all trips"
print("reconciliation with the view: exact")

shape = sql(f"""
    SELECT (SELECT count(*) FROM {SCHEMA}.station_day)                                           AS station_days,
           (SELECT sum(last_day - first_day + 1) FROM {SCHEMA}.station_period)                   AS expected_station_days,
           (SELECT count(DISTINCT station_key) FROM {SCHEMA}.station_day)                        AS stations_in_station_day,
           (SELECT count(*) FROM {SCHEMA}.station_dim)                                           AS stations_in_dim,
           (SELECT count(*) FROM {SCHEMA}.station_day WHERE departures = 0)                      AS quiet_days,
           (SELECT count(*) FROM {SCHEMA}.city_day)                                              AS city_days,
           (SELECT count(*) FROM {SCHEMA}.city_day WHERE no_trips)                               AS days_without_trips
""").iloc[0].fillna(0).astype("int64")
display(shape.to_frame("count"))
assert shape.station_days == shape.expected_station_days, "station_day rows != days inside the active periods"
assert shape.stations_in_station_day == shape.stations_in_dim, "a station of station_dim has no station_day rows (or vice versa)"
print(f"quiet days: {fmt(shape.quiet_days / shape.station_days * 100, '.2f')} % of the station-days")

sql(f"""
    SELECT percentile_cont(ARRAY[0.1, 0.25, 0.5, 0.75, 0.9, 0.99]) WITHIN GROUP (ORDER BY departures) AS departures_percentiles,
           avg(departures) AS mean_departures, max(departures) AS max_departures
    FROM {SCHEMA}.station_day
""")

1 rows in 3,9 s


,count
sd_departures,212.092.181
sd_arrivals,211.657.199
dep_not_nyc,11.330
arr_not_nyc,30.000
raw_departures,212.103.511
raw_arrivals,211.687.199
trips,212.189.382
no_start,85.871
no_end,502.183


reconciliation with the view: exact
1 rows in 3,0 s


,count
station_days,3.872.821
expected_station_days,3.872.821
stations_in_station_day,2.455
stations_in_dim,2.455
quiet_days,111.008
city_days,2.069
days_without_trips,2


quiet days: 2,87 % of the station-days
1 rows in 2,8 s


,departures_percentiles,mean_departures,max_departures
0,"[3.0, 8.0, 25.0, 72.0, 151.0, 340.0]","54,764",1.380


In [ ]:
sql(f"""
    SELECT c.relname AS table_name, pg_get_userbyid(c.relowner) AS owner,
           has_table_privilege('citibike_reader', c.oid, 'SELECT') AS reader_can_select,
           c.reltuples::bigint AS rows_estimate,
           pg_size_pretty(pg_total_relation_size(c.oid)) AS size
    FROM pg_class c JOIN pg_namespace n ON n.oid = c.relnamespace
    WHERE n.nspname = '{SCHEMA}' AND c.relkind = 'r'
    ORDER BY c.relname
""")

## Appendix: what went wrong on 6 October, and the fix for each

| Symptom | Cause | Fix |
| --- | --- | --- |
| A cell keeps running, no response, nothing visible in `pg_stat_activity` on the server | A dead connection in the kernel (VPN hiccup or connection closed while idle); pg8000 had no read timeout, so it waited forever | `connection.py` now sets a 660 s read timeout (just above the server's 10-minute statement timeout) and `pool_pre_ping=True`; restart the kernel when it happens |
| `cannot execute CREATE TABLE in a read-only transaction` (25006) | The logins have `default_transaction_read_only = on`, so every transaction starts read-only whatever the grants say | `SET TRANSACTION READ WRITE` as the first statement, built into `run()` and `copy_into()`. Server-side alternative: `ALTER ROLE <login> RESET default_transaction_read_only;` |
| `DuckDB does not support modifying Postgres tables` (XX000) on `INSERT ... SELECT FROM citibike.trips`, also via `duckdb.query()` | pg_duckdb 0.3 refuses every writing statement in which DuckDB executes the read | By design: `load()` reads through DuckDB into a DataFrame and writes with `COPY FROM STDIN`. Verified on August 2026: 70.836 rows, 5.240.680 departures, difference with the view 0 |
| `permission denied for schema citibike` when the **write role** reads the view | `citibike_prep_writer` is not a member of `citibike_reader` | Not needed with `load()` (the read runs as the login). If the role must ever read the view: `GRANT citibike_reader TO citibike_prep_writer;` as superuser |
| `reader_can_select` false in the last check | The table is not owned by the write role, or the default privileges are missing | `ALTER DEFAULT PRIVILEGES FOR ROLE citibike_prep_writer IN SCHEMA citibike_prep GRANT SELECT ON TABLES TO citibike_reader;` plus `GRANT SELECT ON ALL TABLES IN SCHEMA citibike_prep TO citibike_reader;` for existing tables |
| `duplicate key value violates unique constraint` | The same rows were loaded twice without `delete_where` (or without the `TRUNCATE` of 2.3 / 4.2) | Delete the period or truncate, then load again |

In [ ]:
engine.dispose()